# Prepare regions for upload to MorphoSource

We looked into uploading datasets for sharing, see https://github.com/habi/sticklebacks-manuscript/issues/11
We figured out, that [MorphoSource](https://www.morphosource.org/) is probably the best thing to try.
This notebook is used to prepare the `rec_regions` exports written by the `BucketSeparator.ipynb` notebook for upload to there.

The cells below are used to set up the whole notebook.
They load needed libraries and set some default values.

In [ ]:
# Load the modules we need
import platform
import os
import glob
import pandas
import numpy as np
import scipy
import dask
import dask.array
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from tqdm.auto import tqdm

In [ ]:
# Load our own log file parsing code
# This is loaded as a submodule to alleviate excessive copy-pasting between *all* projects we do
# See https://github.com/habi/BrukerSkyScanLogfileRuminator for details on its inner workings
import BrukerSkyScanLogfileRuminator.parsing_functions as logparse

In [ ]:
# Set up figure defaults
plt.rc('image', cmap='gray', interpolation='nearest')  # Display all images in b&w and with 'nearest' interpolation
# plt.rcParams['figure.figsize'] = (16 * 0.618, 9 * 0.618)  # Size up figures a bit
plt.rcParams['figure.dpi'] = 300

In [ ]:
# Setup scale bar defaults
plt.rcParams['scalebar.location'] = 'lower right'
plt.rcParams['scalebar.frameon'] = False
plt.rcParams['scalebar.color'] = 'white'

In [ ]:
# Set dask temporary folder
# Do this before creating a client: https://stackoverflow.com/a/62804525/323100
# We use the fast internal SSD for speed reasons

import platform
import tempfile
from pathlib import Path

if platform.system() == "Linux":
    # Check if we mounted the FastSSD, otherwise use the standard tmp folder
    fast_ssd = Path("/media/habi/Fast_SSD")

    if fast_ssd.exists():
        tmp_path = fast_ssd / "tmp"
    else:
        tmp_path = Path(tempfile.gettempdir())

elif platform.system() == "Darwin":
    tmp_path = Path(tempfile.gettempdir())

elif platform.system() == "Windows":
    if "anaklin" in platform.node():
        tmp_path = Path(r"F:\tmp")
    else:
        tmp_path = Path(r"D:\tmp")

else:
    raise RuntimeError(f"Unsupported operating system: {platform.system()}")

tmp_path.mkdir(parents=True, exist_ok=True)

dask.config.set({"temporary_directory": tmp_path})

print(f"Dask temporary files go to {dask.config.get('temporary_directory')}")

In [ ]:
from dask.distributed import Client
client = Client()
client

Since the (tomographic) data can reside on different drives we set a folder to use below

In [ ]:
# Different locations if running either on Linux or Windows
FastSSD = True
if 'Linux' in platform.system():
    if FastSSD:
        BasePath = os.path.join(os.sep, 'media', 'habi', 'Fast_SSD')
    else:
        BasePath = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
elif 'Windows' in platform.system():
    if FastSSD:
        BasePath = os.path.join('F:\\')
    else:
        BasePath = os.path.join('N:\\')
if 'research_storage_ben' in BasePath:
    Root = os.path.join(BasePath)
else:
    Root = os.path.join(BasePath, 'IEE Stickleback')
# Force reading from Bens research storage folder
# Root = os.path.join(os.path.sep, 'home', 'habi', 'research_storage_ben', 'microCT_Stickleback')
print('We are loading all the data from %s' % Root)
Root = Path(Root)

Now that we are set up, actually start to load/ingest the data.

In [ ]:
# Make us a dataframe for saving all that we need
Data = pandas.DataFrame()

In [ ]:
# Get *all* log files present on disk
# Using os.walk is way faster than using recursive glob.glob
# Not sorting the found logfiles is also making it quicker
Data['LogFile'] = [os.path.join(root, name)
                   for root, dirs, files in os.walk(Root)
                   for name in files
                   if name.endswith((".log"))]

In [ ]:
# Get all folders
Data['Folder'] = [os.path.dirname(f) for f in Data['LogFile']]

In [ ]:
# Generate us a shorter folder name for easier printing
Data['FolderShort'] = [f[len(Root.name) + 1:] for f in Data['Folder']]

In [ ]:
# Show a (small) sampler of the loaded data as a first check
Data.sample(n=5)

In [ ]:
# Check for samples which are not yet reconstructed
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row.Folder and '_14/proj2' not in row.Folder and '_15/proj2' not in row.Folder:
        # Sticklebucket_14/proj2/Sticklebucket_14~00.log and 
        # Sticklebucket_15/proj2/Sticklebucket_15~00.log are failed scans where we cannot do a reconstruction, so we exclude them
        # If there's nothing with 'rec*' on the same level, then tell us
        if not glob.glob(row.Folder.replace('proj', '*rec*')):
            print('- %s is missing matching reconstructions' % row.LogFile[len(Root.name) + 1:])

In [ ]:
# Search for any .csv files in each folder.
# These are only generated when the "X/Y Alignment With a Reference Scan" was performed in NRecon.
# If those files do *not* exist we have missed to do it and should correct for this.
Data['XYAlignment'] = [glob.glob(os.path.join(f, '*T*.csv')) for f in Data['Folder']]

In [ ]:
# Display samples which are missing the .csv-files for the XY-alignment
for c, row in Data.iterrows():
    # Iterate over every 'proj' folder
    if 'proj' in row['Folder']:
        if not row['XYAlignment']:
            if not any(x in row.LogFile for x in ['rectmp.log',  # because we only exclude temporary logfiles in a later step
                                                  '14/proj',  # *All* scans of Sticklebucket 14 are missing the alignment files and cannot be aligned. 
                                                  '15/proj',  # *All* scans of Sticklebucket 15 are missing the alignment files and cannot be aligned
                                                  ]):
                print('- %s has *not* been X/Y aligned' % row.LogFile[len(Root.name) + 1:])

In [ ]:
# Get rid of all the logfiles from all the folders that might be on disk but that we don't want to load the data from
for c, row in Data.iterrows():
    if 'ucket' not in row.Folder:  # Only use the scans named Bucket* here, e.g. BucketOfFish_* and Sticklebucket_*
        Data.drop([c], inplace=True)
    elif 'rec' not in row.Folder:  # Only look at logs in the rec folders
        Data.drop([c], inplace=True)
    elif '_regions' in row.Folder:  # Exclude all log files that we write in this notebook (to $scan$_region folders)
        Data.drop([c], inplace=True)
    elif os.path.split(row.LogFile)[1].startswith('._'):  # Remove macos metadata files for files on external storage
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '15um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '18um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'BucketOfFish_A' in row.LogFile and '19um_rec' in row.LogFile:  # BucketOfFish_A is the first bucket. For that one we did some test scans. Only use 17.5_ 
        Data.drop([c], inplace=True)
    elif 'Sticklebucket_14' in row.LogFile and 'rec3' in row.LogFile:  # Sticklebucket_14/rec3 is a repeat scan of Sticklebucket_14/rec. Sheila and Ben took all the data for the manuscript from the rec scan though, hence do not look at the rec3 scan here
        Data.drop([c], inplace=True)        
    elif 'Sticklebucket_15' in row.LogFile and 'rec3' in row.LogFile:  # Sticklebucket_14/rec3 is a repeat scan of Sticklebucket_14/rec. Sheila and Ben took all the data for the manuscript from the rec scan though, hence do not look at the rec3 scan here
        Data.drop([c], inplace=True)                
# Reset dataframe to something that we would get if we only would have loaded the 'rec' files
Data = Data.reset_index(drop=True)

It's a bit silly to exclude the self-written log files (`*_regions/*/*.log`) above, but like so we know for sure to include everything we've exported...

In [ ]:
# Generate us some meaningful colums in the dataframe
Data['Sample'] = [os.path.basename(log).replace('_rec.log', '') for log in Data['LogFile']]
Data['Scan'] = [os.path.basename(os.path.dirname(log)) for log in Data['LogFile']]

In [ ]:
# Show the data from the last loaded scans
Data.tail(n=5)

In [ ]:
# Load the file names of all the reconstructions of all the scans
Data['Filenames Reconstructions'] = [sorted(glob.glob(os.path.join(f, '*rec0*.png'))) for f in Data['Folder']]
# How many reconstructions do we have?
Data['Number of reconstructions'] = [len(r) for r in Data['Filenames Reconstructions']]

In [ ]:
# Drop samples which have either not been reconstructed yet or of which we deleted the reconstructions with
# `find . -name "*rec*.png" -type f -mtime +333 -delete`
# Based on https://stackoverflow.com/a/13851602
# for c,row in Data.iterrows():
#     if not row['Number of reconstructions']:
#         print('%s contains no PNG files, we might be currently reconstructing it' % row.Folder)
Data = Data[Data['Number of reconstructions'] > 0]
# Reset the dataframe count/index for easier indexing afterwards
Data.reset_index(drop=True, inplace=True)
print('We have %s folders with reconstructions' % (len(Data)))

In [ ]:
# Get parameters we need to submit to MorphoSource from the log files
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['ControlSoftware'] = [logparse.controlsoftware(log) for log in Data['LogFile']]
Data['Source'] = [logparse.source(log) for log in Data['LogFile']]
Data['Detector'] = [logparse.camera(log) for log in Data['LogFile']]
Data['DetectorVoxelsize'] = [logparse.cameravoxelsize(log) for log in Data['LogFile']]
Data['Voxelsize'] = [logparse.pixelsize(log) for log in Data['LogFile']]
Data['Filter'] = [logparse.whichfilter(log) for log in Data['LogFile']]
Data['Exposuretime'] = [logparse.exposuretime(log) for log in Data['LogFile']]
Data['Scanner'] = [logparse.scanner(log) for log in Data['LogFile']]
Data['NumProj'] = [logparse.numproj(log) for log in Data['LogFile']]

Data['Voltage'] = [logparse.voltage(log) for log in Data['LogFile']]
Data['Power'] = [logparse.power(log) for log in Data['LogFile']]
Data['Current'] = [logparse.current(log) for log in Data['LogFile']]

Data['Source object distance'] = [logparse.distance_source_to_sample(log) for log in Data['LogFile']]
Data['Source detector distance'] = [logparse.distance_source_to_detector(log) for log in Data['LogFile']]

Data['Averaging'] = [logparse.averaging(log) for log in Data['LogFile']]
Data['ProjectionSize'] = [logparse.projection_size(log) for log in Data['LogFile']]
Data['Stacks'] = [logparse.stacks(log) for log in Data['LogFile']]
Data['RotationStep'] = [logparse.rotationstep(log) for log in Data['LogFile']]
Data['Grayvalue'] = [logparse.reconstruction_grayvalue(log) for log in Data['LogFile']]
Data['RingartefactCorrection'] = [logparse.ringremoval(log) for log in Data['LogFile']]
Data['BeamHardeningCorrection'] = [logparse.beamhardening(log) for log in Data['LogFile']]
Data['DefectPixelMasking'] = [logparse.defectpixelmasking(log) for log in Data['LogFile']]
Data['Scan date'] = [logparse.scandate(log) for log in Data['LogFile']]

In [ ]:
# Sort dataframe based on the scan date
Data.sort_values(by=['Scan date'],
                 ignore_index=True,
                 inplace=True)

Since we've done everything *correctly* in `BucketSeparator.ipynb` we can simply go through all the desired folders and pull all in from disk.
This is more efficient than re-doing the extraction from scratch :)

In [ ]:
# Construct folder name for regions folder
Data['FolderRegionsExports'] = None
for c, row in Data.iterrows():
    Data.at[c, 'FolderRegionsExports'] = os.path.join(os.path.dirname(os.path.dirname(row.LogFile)), row.Scan + '_regions')

In [ ]:
# Search for log files we've written
Data['RegionsLogFiles'] = [glob.glob(os.path.join(folder, '*', '*.log')) for folder in Data['FolderRegionsExports']]

In [ ]:
# Construct regions name (and double-check for errors on the way)
Data['RegionsName'] = None
Data['RegionsFolder'] = None
for c, row in Data.iterrows():
    Data.at[c, 'RegionsName'] = [os.path.splitext(os.path.basename(logfilename))[0] for logfilename in row.RegionsLogFiles]
    Data.at[c, 'RegionsFolder'] = [os.path.dirname(logfilename) for logfilename in row.RegionsLogFiles]
    for rn, rf in zip(Data.at[c, 'RegionsName'], Data.at[c, 'RegionsFolder']):
        if rn != os.path.basename(rf):  # Tested with a manual rename on disk :)
            print(f'Error: For {row.LogFile[len(Root):]}: Extracted Region name "{rn}" does not match extracted folder name "{os.path.basename(rf)}"')

In [ ]:
# We should have 215+ regions, as this includes everything.
# Later we exclude 'WK.X24.001', as according to Ben, "[w]e also excluded Wik lake from all samples (as it had n =1)".
print('We have %s regions in total' % len(Data.RegionsName.explode()))

MorphoSource would like to ingest a ".zip containing .tif, .jpeg, .bmp, or .dcm*", see https://docs.google.com/document/d/1QByWl5t0SFD4QkdxUdoUbeTNms3HEhQYdTndo6PR-Ts/edit?tab=t.0, so we're preparing these files.
In [a test](https://www.morphosource.org/concern/media/000885110?locale=en), we've seen that a .zip with PNGs works fine, too...

Each .zip file should contain the original `proj/*.log`, `rec/*.log` and all the files from `rec_regions/FishID/*` for reproducible research.

In [ ]:
# Define us a "custom" zipping function
import pathlib
import zipfile

def zip_folder(folder, logfile, scan_date):
    # Generate folder names
    folder = pathlib.Path(folder)
    logfile = pathlib.Path(logfile)

    # Generate path for the zip file
    zip_path = folder.parent / (folder.name + ".zip")

    # Don't regenerate an existing ZIP
    if zip_path.exists():
        return zip_path

    # Search for correct label-checking file
    search_string = folder.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        folder.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )

    # Search for file in which we've specified the positions of the fish for the second batch of scans
    regionmdfile = next(
        folder.parent.parent.glob('*.Mapping*Region.md'),
        None
    )

    # Create README file (with function below)
    readme_path = create_readme(folder, logfile, scan_date)

    # Actually do the zipping now
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        # Add region folder
        for file in folder.rglob("*"):
            zf.write(file, arcname=file.relative_to(folder.parent))

        # Add reconstruction log file
        zf.write(logfile, arcname=logfile.name)

        # Add label-checking file
        zf.write(labelcheckingfile, arcname=labelcheckingfile.name)

        if regionmdfile is not None:
            # Add region metadata file
            zf.write(regionmdfile, arcname=regionmdfile.name)

        # Add README
        zf.write(readme_path, arcname="README.md")

    # Remove temporary README
    readme_path.unlink()

    return zip_path

In [ ]:
def get_file_listing(region):
    from pathlib import Path
   
    region = Path(region)

    files = sorted([f.relative_to(region) for f in region.rglob("*") if f.is_file()])

    pngs = [f for f in files if f.suffix.lower() == ".png"]
    regionlog = [f for f in files if f.suffix.lower() == ".log"]

    return pngs, regionlog

In [ ]:
# We want to add a custom/dynamic README.md file to *every* archive, so let's generate one
from datetime import datetime

def create_readme(region, logfile, scan_date):
    region = pathlib.Path(region)
    logfile = pathlib.Path(logfile)
    pngs, regionlog = get_file_listing(region)

    if len(pngs) > 2:
        png_listing = (
            f"│   ├── {pngs[0]}\n"
            f"│   ├── {pngs[1]}\n"
            f"│   ├── ...\n"
            f"│   ├── {pngs[-1]}"
        )
    elif len(pngs) == 1:
        png_listing = "\n".join(f"│   ├── {p}" for p in pngs)
    else:
        png_listing = ""

    log_listing = "\n".join(f"│   └── {l}" for l in regionlog)

    # Search for correct label-checking file
    search_string = region.parent.name.replace("_regions", "")
    labelcheckingfile = next(
        region.parent.parent.glob(f"*{search_string}*.Labels.Check.png"),
        None
    )    

    readme = f"""# {region.name}

## Info

This archive was generated on {datetime.now().isoformat(timespec="seconds")} with [a bespoke notebook](https://github.com/habi/sticklebacks-manuscript/PrepareForMorphoSource.ipynb) for uploading extracted datasets from [our sticklebacks manuscript](https://habi.github.io/sticklebacks-manuscript/) to [MorphoSource](https://www.morphosource.org/).
It contains the {len(pngs)} cropped reconstructions for specimen **{region.name}**, which was scanned on {scan_date}, and some aditional files.

## Archive contents

```console
{region.name}.zip
├── {region.name}/
{png_listing}
{log_listing}
├── {logfile.name}
├── {labelcheckingfile.name}
└── README.md
```

## Source

- `{region.name}`: Original region folder on disk from `{region.relative_to(Root)}`
- `{logfile.name}`: Log file of the reconstructions of the original multi-specimen scan copied into the archive from `{logfile.relative_to(Root)}`
- `{labelcheckingfile.name}`: Label/vial checking file generated from the the original multi-specimen scan. Generated with [the separator notebook](https://github.com/habi/sticklebacks/blob/main/BucketSeparator.ipynb) and copied into the archive from `{labelcheckingfile.relative_to(Root)}`.
- `README.md`: This file.
"""

    readme_path = region.parent / "README.md"
    readme_path.write_text(readme, encoding="utf-8")

    return readme_path

In [ ]:
for c, row in tqdm(Data.iterrows(), desc='Zipping', total=len(Data)):
    for d, region in tqdm(enumerate(row.RegionsFolder),
                          desc=f'Zipping regions of {row.FolderShort}',
                          total=len(row.RegionsFolder),
                          leave=False):
        if 'WK.X24.001' not in region: 
            # Skip the one specimen that was excluded from the manuscript because it was a singleton
            # According to Ben, "[w]e also excluded Wik lake from all samples (as it had n =1)".
            zip_folder(region, row.LogFile, row['Scan date'])
        else:
            print(f'Skipping {region} because it was excluded from the manuscript as a singleton')

In [ ]:
# Generate zip file names for each fish
Data['ZipFileName'] = [fishid + '.zip' for fishid in Data['Fish_ID']]

In [ ]:
print(f"We now have {len(Data['ZipFileName'].unique())} zip files on disk.")

---
Now we have all the .zip files ready for MorphoSource.
We've uploaded all of them with [Globus](https://globus.org/), after they "generated" a folder for us.
The batch upload needs a "submission manifest Excel file (XLSX)", as mentioned on https://duke.atlassian.net/wiki/spaces/MD/pages/35423526/Batch+Submitting+Multiple+Media
We were able to download that file manually (`ingest_manifest_template-e1df68c6f6f057152750981f74d32b2705eab8f7945c5b60d4d7e76b9d94aa35.xlsx`) in the dashboard at MorphoSource after 'requesting a batch upload' there.
To properly fill this files, we need some more data, which we pull from different sources into our dataframe.
At the end we're writing the correct colums to the file.

In [ ]:
# MorphoSource would like to have the sex data.
# For the 2023 fish, this is unknown, for the 2024 fish, this is in `2024_Fish_Data_Lynn.csv`
# In there, we have the 'Unique ID' column, which is the same as the FishID column in our dataframe.
SexData = pandas.read_csv(os.path.join(Root, '2024_Fish_Data_Lynn.csv'))
# Merge the sex data into the MorphoSource dataframe
Data = Data.merge(SexData[['Unique_ID', 'Sex']], left_on='Fish_ID', right_on='Unique_ID', how='left')
# Expand F and M to Female and Male, respectively
Data['Sex'] = Data['Sex'].replace({'F': 'Female', 'M': 'Male'})
# Fill empty values in the Sex column, so they are not 'none' in the CSV export later on
Data['Sex'] = Data['Sex'].fillna("Undetermined")


In [ ]:
# MorphoSource can also ingest lat/lon
# Accordint to Ben, this is
# --
# Watson: 60.539000, -150.467000
# Finger: 61.605600, -149.279200
# Spirit: 60.596104, -150.997664
# South Rolly: 61.667545, -150.135689
# Walby: 61.620000, -149.213000
# Tern: 60.533128, -149.547014
# --
# First construct the lake name (which we can add to MorphoSource as locality)
Data['LakeShort'] = [fishid.split('.')[0] for fishid in Data['Fish_ID']]
lake_names = {
    "FG": "Finger Lake",
    "SL": "Spirit Lake",
    "SR": "South Rolly Lake",
    "WT": "Watson Lake",
    "TL": "Tern Lake",
    "WB": "Walby Lake",
}
Data['Lake'] = [lake_names.get(lake, 'Unknown') for lake in Data['LakeShort']]
# Construct lat/lon columns based on the lake name
lat_lon = {
    "Watson Lake": (60.539000, -150.467000),
    "Finger Lake": (61.605600, -149.279200),
    "Spirit Lake": (60.596104, -150.997664),
    "South Rolly Lake": (61.667545, -150.135689),
    "Walby Lake": (61.620000, -149.213000),
    "Tern Lake": (60.533128, -149.547014)
}
Data['LatLon'] = [lat_lon.get(lake, (0, 0)) for lake in Data['Lake']]
Data['Latitude'] = [latlon[0] for latlon in Data['LatLon']]
Data['Longitude'] = [latlon[1] for latlon in Data['LatLon']]

In [ ]:
# Populate dataframe with values that are equal for all scans
Data['Project'] = 'https://www.morphosource.org/projects/000885106'
Data['Species'] = 'Gasterosteus aculeatus'
Data['Object organization'] = 'https://www.morphosource.org/organizations/000898902'
Data['Creator'] = 'https://www.morphosource.org/users/b4a341'  # Used in 'imaging'
Data['Object element or part'] = 'Full body'
Data['Shading correction'] = True
Data['Surrounding material'] = 'Basotect (melamine resin foam)'
Data['Target type'] = 'Transmission'
Data['Detector type'] = 'Direct (X-Ray photoconductor)'
Data['Detector configuration'] = 'Area (single or tiled detector)'
Data['Target material'] = 'Tungsten'  # https://www.hamamatsu.com/content/dam/hamamatsu-photonics/sites/documents/99_SALES_LIBRARY/etd/L10711_TXPR1039E.pdf
Data['Rotation number'] = 'unknown'
Data['Phase contrast'] = False
Data['Optical magnification'] = False
Data['Acquisition type'] = 'Sequenced (Rotational)'

In [ ]:
# Some fish are not 'Full body'
# Let's change that based on the length of the cropped reconstructions and double-check with the MIP.
# First see which ones...
Data['NumRecCropped'] = [len(r) for r in Data['Reconstructions_Cropped']]

In [ ]:
Data[['Fish_ID', 'NumRecCropped', 'Slice_Start', 'Slice_End']].sort_values(by='NumRecCropped', ascending=True).head(n=15)

In [ ]:
# Let's see which ones are only heads
for c, row in Data.sort_values(by='NumRecCropped', ascending=True).head(n=15).iterrows():
    plt.imshow(row.MIP)
    plt.axvline(row.Slice_Start, color='green', linestyle='-', label='Start crop')
    plt.axvline(row.Slice_End, color='blue', linestyle='-', label='End crop')
    plt.title(f'Fish ID: {row.Fish_ID}, NumRecCropped: {row.NumRecCropped}')
    plt.gca().add_artist(ScaleBar(row['Voxelsize'], 'um'))
    plt.show()
# So, if we have less than 1900 cropped reconstructions, we can assume that it's not a full body scan and change the 'Object element or part' column accordingly.
Data.loc[Data['NumRecCropped'] < 1900, 'Object element or part'] = 'Head'

In [ ]:
# Massage other data in the dataframe to MorphoSource naming
# Imaging
Data.rename(columns={'Scan date': 'Event date'}, inplace=True)
Data['Software'] = [' '.join([scnr.replace(' ', ''), 'Control Program', f'(version {swv})']) for scnr, swv in zip(Data['Scanner'], Data['ControlSoftware'])]
Data.rename(columns={'Exposuretime': 'Exposure time'}, inplace=True)
Data.rename(columns={'Averaging': 'Frame averaging'}, inplace=True)
Data.rename(columns={'NumProj': 'Projections'}, inplace=True)
Data.rename(columns={'Current': 'Amperage'}, inplace=True)
Data.rename(columns={'Source': 'X-ray tube type'}, inplace=True)
Data['Detector pixels X'] = [ps[0] for ps in Data['ProjectionSize']]
Data['Detector pixels Y'] = [ps[1] for ps in Data['ProjectionSize']]
Data['Detector pixels size X'] = Data['DetectorVoxelsize']
Data['Detector pixels size Y'] = Data['DetectorVoxelsize']

The MorphoSource team asked us to *manually* upload 5 `.zip` files.
Surface the data of those for copy-pasting.

In [ ]:
manual = ['FG.X24.001',
          'SL.X24.001',
          'TL.X24.001',
          'WB.X24.001',
          'WT.X24.001']
fish_id_to_find = manual[4]

In [ ]:
for i, row in Data[Data['Fish_ID'] == fish_id_to_find][[
    'Fish_ID', 'Sex', 'Lake', 'Latitude', 'Longitude',
    'Scanner', 'Filter', 'Exposure time', 'Shading correction', 'Frame averaging',
    'Projections', 'Voltage', 'Power', 'Amperage', 'Surrounding material',
    'X-ray tube type', 'Target type', 'Detector type',
    'Detector pixels X', 'Detector pixels size X', 'Detector pixels Y', 'Detector pixels size Y',
    'Detector configuration', 'Source object distance', 'Source detector distance', 'Target material', 'Rotation number', 'Phase contrast', 'Optical magnification',
    'Acquisition type'
]].iterrows():

    print(f"\n--- {row['Fish_ID']} ---")
    for key, value in row.items():
        print(f"{key:25}: {value}")

In [ ]:
# Drop manually uploaded from dataframe
for fish_id in manual:
    Data = Data.drop(Data[Data.Fish_ID == fish_id].index)

**NOW** construct us the final dataframe to save to XLS.

In [ ]:
# Read in the file from MorphoSource
# We downloaded the template file and preprended 'original.' to it.
xlsfilename = glob.glob(os.path.join('original.ingest_manifest_template-*.xlsx'))[0]
MSXLS = pandas.read_excel(xlsfilename, header=None)
MSXLS.columns = MSXLS.iloc[6] # MorphoSource writes their machine-readable colum names in row 7 (index 6), so we use that as the header

In [ ]:
# Go through the MorphoSource XLSX file and tell us which columns are "Required" in row 2 (Required/Recommended)
required_columns = MSXLS.iloc[3] == 'Required'
print (f"Required columns in the MorphoSource XLSX file: {list(MSXLS.columns[required_columns])}")
for col in MSXLS.columns[required_columns]:
    print(80*'-')
    print(col)
    print(f"**Definition**: {MSXLS[col].iloc[2]}")
    print(f"**Controlled vocabulary**: {MSXLS[col].iloc[5]}")
    print(f"**Example**: {MSXLS[col].iloc[6:].values[1:]}")

In [ ]:
# Now that we know what we *need* and *can* fill in, rename the needed and recommended colums in our dataframe
# Required
Data.rename(columns={'ZipFileName': 'media.media_file'}, inplace=True)
Data['media.media_type'] = 'CTImageSeries'
Data['media.raw_or_derived'] = 'derived'

In [ ]:
# Go through the MorphoSource XLSX file and tell us which columns are "Recommended" in row 2 (Required/Recommended)
recommended_columns = MSXLS.iloc[3] == 'Recommended'
print (f"Recommended columns in the MorphoSource XLSX file: {list(MSXLS.columns[recommended_columns])}")
for col in MSXLS.columns[recommended_columns]:
    print(80*'-')
    print(col)
    print(f"**Definition**: {MSXLS[col].iloc[2]}")
    print(f"**Controlled vocabulary**: {MSXLS[col].iloc[5]}")
    print(f"**Example**: {MSXLS[col].iloc[6:].values[1]}")

In [ ]:
# Now that we know what we *need* and *can* fill in, rename the needed and recommended colums in our dataframe
# Recommended
Data.rename(columns={'Fish_ID': 'biological_specimen.catalog_number'}, inplace=True)
Data.rename(columns={'Object element or part': 'media.part'}, inplace=True)

In [ ]:
# Fill/rename media
Data['media.creator'] = 'David Haberthür'
Data.rename(columns={'Event date': 'media.date_created'}, inplace=True)
Data['media.date_created'] = [dc.strftime("%Y-%m-%d") for dc in pandas.to_datetime(Data['media.date_created'])]  # MorphoSource wants YYYY-MM-DD 
Data.rename(columns={'Voxelsize': 'media.x_spacing'}, inplace=True)
Data['media.y_spacing'] = Data['media.x_spacing']  # we have isometric voxels
Data['media.z_spacing'] = Data['media.x_spacing']  # we have isometric voxels
Data['media.unit'] = 'um'
Data['media.series_type'] = 'Reconstructed image stack'

In [ ]:
# Fill/rename biological
Data['biological_specimen.date_created'] = ['2023' if 'X23' in fishid else '2024' for fishid in Data['biological_specimen.catalog_number']]
Data['biological_specimen.date_created'] = [dc.strftime("%Y-%m-%d") for dc in pandas.to_datetime(Data['biological_specimen.date_created'])]  # MorphoSource wants YYYY-MM-DD 
Data['biological_specimen.description'] = 'Gasterosteus aculeatus'
Data['biological_specimen.institution_code'] = 'IEE'
Data['biological_specimen.collection_code'] = 'FITNESS'
Data['biological_specimen.creator'] = 'R. Benjamin Sulser, Sheila Christen, Catherine L. Peichel'
Data.rename(columns={'Lake': 'biological_specimen.original_location'}, inplace=True)
Data.rename(columns={'Latitude': 'biological_specimen.latitude'}, inplace=True)
Data.rename(columns={'Longitude': 'biological_specimen.longitude'}, inplace=True)
Data.rename(columns={'Sex': 'biological_specimen.sex'}, inplace=True)
Data['biological_specimen.vouchered'] = 'yes'

In [ ]:
# Fill/rename taxonomy columns
Data['taxonomy.taxonomy_genus'] = 'Gasterosteus'
Data['taxonomy.taxonomy_species'] = 'G. aculeatus'

In [ ]:
# Fill/rename imaging
Data['imaging_event.description'] = 'X-ray microtomography (microCT) scan of a stickleback fish'
Data['imaging_event.creator'] = Data['media.creator']
Data['imaging_event.software'] = [f'{scanner.replace(' ', '')} Control Program (version {swv})' for scanner, swv in zip(Data['Scanner'], Data['ControlSoftware'])]
Data['imaging_event.date_created'] = Data['media.date_created']
Data.rename(columns={'Exposure time': 'imaging_event.ct.exposure_time'}, inplace=True)
Data['imaging_event.ct.shading_correction'] = 'yes'
Data.rename(columns={'Filter': 'imaging_event.ct.ie_filter'}, inplace=True)
Data.rename(columns={'Frame averaging': 'imaging_event.ct.frame_averaging'}, inplace=True)
Data.rename(columns={'Projections': 'imaging_event.ct.projections'}, inplace=True)
Data.rename(columns={'Stacks': 'imaging_event.ct.rotation_number'}, inplace=True)
Data.rename(columns={'Voltage': 'imaging_event.ct.voltage'}, inplace=True)
Data.rename(columns={'Power': 'imaging_event.ct.power'}, inplace=True)
Data.rename(columns={'Amperage': 'imaging_event.ct.amperage'}, inplace=True)
Data.rename(columns={'Surrounding material': 'imaging_event.ct.surrounding_material'}, inplace=True)
Data.rename(columns={'X-ray tube type': 'imaging_event.ct.xray_tube_type'}, inplace=True)
Data.rename(columns={'Target type': 'imaging_event.ct.target_type'}, inplace=True)
Data.rename(columns={'Detector type': 'imaging_event.ct.detector_type'}, inplace=True)
Data.rename(columns={'Detector pixels X': 'imaging_event.ct.detector_pixels_x'}, inplace=True)
Data.rename(columns={'Detector pixels size X': 'imaging_event.ct.detector_pixel_size_x'}, inplace=True)
Data.rename(columns={'Detector pixels Y': 'imaging_event.ct.detector_pixels_y'}, inplace=True)
Data.rename(columns={'Detector pixels size Y': 'imaging_event.ct.detector_pixel_size_y'}, inplace=True)
Data.rename(columns={'Detector configuration': 'imaging_event.ct.detector_configuration'}, inplace=True)
Data.rename(columns={'Source object distance': 'imaging_event.ct.source_object_distance'}, inplace=True)
Data.rename(columns={'Source detector distance': 'imaging_event.ct.source_detector_distance'}, inplace=True)
Data.rename(columns={'Target material': 'imaging_event.ct.target_material'}, inplace=True)
Data.rename(columns={'Target material': 'imaging_event.ct.rotation_number'}, inplace=True)
Data['imaging_event.ct.phase_contrast'] = False
Data['imaging_event.ct.optical_magnification'] = False
Data['imaging_event.ct.acquisition_type'] = 'ConstantAngle'

In [ ]:
# Fill/rename processing columns
Data['processing_event.creator'] = 'David Haberthür'
Data['processing_event.software'] = 'https://doi.org/10.5281/zenodo.18257527'
Data['processing_event.description'] = 'https://github.com/habi/sticklebacks-manuscript'

In [ ]:
# Drop example rows from the MorphoSource XLSX file
MSXLS.drop([7, 8], inplace=True)
MSXLS.head(n=10)

In [ ]:
# Get all colums from the MSXLS dataframe
columns = MSXLS.columns.intersection(Data.columns)
# Concatenate the Data dataframe with the MSXLS dataframe, but only keep the columns that are in the MSXLS dataframe
MSXLS = pandas.concat([MSXLS, Data[columns]], ignore_index=True)

In [ ]:
# What do we have now?
MSXLS.head(n=15)

In [ ]:
# Save out to an new XLSX file, which we can then upload to MorphoSource
MSXLS.to_excel(xlsfilename.replace('original.', ''), index=False, header=False)